In [1]:
# Syed Hussain Mehdi Raza
# 25K-9016
# Assignment 2
# Deep Learning
# YOLO

In [8]:
!pip -q install ultralytics pycocotools tqdm pyyaml pandas seaborn

import json
import shutil
import zipfile
from pathlib import Path
from typing import Dict, List, Tuple
from urllib.request import urlretrieve

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
import yaml
import requests

from ultralytics import YOLO

plt.style.use('seaborn-v0_8')
np.random.seed(259016)

In [3]:
#  My ID is 9016
#  Last 2 digits are 16
#  Modulo 10 is 6
#  + 5
#  N is 11
N = 11

EPOCHS = 5 # Not having better GPU or paid google colab, so 5 epochs sir 🙏
IMG_SIZE = 640
BATCH = 16
MODEL_NAME = 'yolov8n.pt'

TRAINING_DATA_URL = 'http://images.cocodataset.org/zips/train2017.zip'
VALIDATION_DATA_URL = 'http://images.cocodataset.org/zips/val2017.zip'
ANNOTATIONS_DATA_URL = 'http://images.cocodataset.org/annotations/annotations_trainval2017.zip'


ROOT = Path('/content')
DATA_ROOT = ROOT / 'datasets'
COCO_ROOT = DATA_ROOT / 'coco2017'
SUBSET_ROOT = DATA_ROOT / f'coco_subset_n{N}'
RUNS_ROOT = ROOT / 'runs'

print(f'N={N}')
print(f'COCO_ROOT={COCO_ROOT}')
print(f'SUBSET_ROOT={SUBSET_ROOT}')

N=11
COCO_ROOT=/content/datasets/coco2017
SUBSET_ROOT=/content/datasets/coco_subset_n11


In [18]:
# Download and extract dataset functions
# tqdm and ipywidgets for download progress

def ensure_dir(path: Path) -> None:
    path.mkdir(parents=True, exist_ok=True)

def download_file(url, filename: Path):
    """
    Downloads a file from a URL with a progress bar in a Jupyter Notebook.
    """
    ensure_dir(filename.parent)
    response = requests.get(url, stream=True, allow_redirects=True)
    if response.status_code != 200:
        raise RuntimeError(f"Request to {url} returned status code {response.status_code}")

    # Get the total file size from the response headers
    total_size = int(response.headers.get("content-length", 0))
    block_size = 1024 # 1 KB chunks

    # Use tqdm as a context manager to display the progress bar
    with tqdm(total=total_size, unit="B", unit_scale=True, desc=str(filename), leave=True) as progress_bar:
        with open(filename, "wb") as file:
            for data in response.iter_content(block_size):
                progress_bar.update(len(data))
                file.write(data)

    if total_size != 0 and progress_bar.n != total_size:
        print("Warning: Downloaded file size does not match expected size")

def extract_zip(zip_path: Path, target_dir: Path, expected_path: Path = None) -> None:
    if expected_path is not None and expected_path.exists():
        print(f'[skip] already extracted: {expected_path}')
        return

    print(f'[extract] {zip_path.name} -> {target_dir}')
    with zipfile.ZipFile(zip_path, 'r') as zf:
        ensure_dir(target_dir)
        members = zf.infolist()
        with tqdm(total=len(members), unit="file", desc=str(zip_path.name), leave=True) as progress_bar:
            for member in members:
                zf.extract(member, target_dir)
                progress_bar.update(1)
    print('[done] extraction complete')

In [19]:
# Downloading start, step by step
#  Annotations first small zip
annotations_zip = COCO_ROOT / 'annotations_trainval2017.zip'
download_file(ANNOTATIONS_DATA_URL, annotations_zip);

/content/datasets/coco2017/annotations_trainval2017.zip:   0%|          | 0.00/253M [00:00<?, ?B/s]

In [20]:
# Download Training Data
train_zip = COCO_ROOT / 'train2017.zip'
download_file(TRAINING_DATA_URL, train_zip);

/content/datasets/coco2017/train2017.zip:   0%|          | 0.00/19.3G [00:00<?, ?B/s]

In [21]:
# Download validation Data
val_zip = COCO_ROOT / 'val2017.zip'
download_file(VALIDATION_DATA_URL, val_zip);

/content/datasets/coco2017/val2017.zip:   0%|          | 0.00/816M [00:00<?, ?B/s]

In [22]:
# Extraction Time
extract_zip(annotations_zip, COCO_ROOT)

[extract] annotations_trainval2017.zip -> /content/datasets/coco2017
[done] extraction complete


In [ ]:
extract_zip(train_zip, COCO_ROOT)

[extract] train2017.zip -> /content/datasets/coco2017


Exception ignored in: <function tqdm.__del__ at 0x7e7a9c52c220>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/tqdm/std.py", line 1148, in __del__
    self.close()
  File "/usr/local/lib/python3.12/dist-packages/tqdm/notebook.py", line 277, in close
    self.disp(bar_style='danger', check_delay=False)
    ^^^^^^^^^
AttributeError: 'tqdm' object has no attribute 'disp'
